In [1]:
from llama_cpp import Llama
from llama_cpp.llama_types import ChatCompletionRequestUserMessage

import pytesseract
import datetime
from pdf2image import convert_from_path
from PIL import Image, ImageChops
import os

In [2]:
# Load the model
llm = Llama.from_pretrained(
    repo_id="unsloth/Qwen3-1.7B-GGUF",
    filename="*Q4_K_M.gguf",   # wildcard picks the matching file automatically
    n_ctx=10245,                 # context window size
    n_gpu_layers=-1,            # -1 = offload all layers to GPU, 0 = CPU only
    verbose=False
)

llama_context: n_ctx_per_seq (10245) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


In [3]:
directory_name = "resume_samples_pdf"
img_directory_name = "resume_sample_imgs_concat"
pdf_files = [pdf for pdf in os.listdir(directory_name)]

In [4]:
def image_concat(to_concat_images):

    widths, heights = zip(*(i.size for i in to_concat_images))

    max_width = max(widths)
    total_height = sum(heights)

    new_im = Image.new('RGB', (max_width, total_height))

    y_offset = 0

    for im in to_concat_images:
      new_im.paste(im, (0, y_offset))
      y_offset += im.size[1]

    return new_im
    # new_im.save('test.jpg')

def trim_whitespace(im):
    bg = Image.new(im.mode, im.size, im.getpixel((0,0)))
    diff = ImageChops.difference(im, bg)
    diff = ImageChops.add(diff, diff, 2.0, -100)
    bbox = diff.getbbox()
    if bbox:
        return im.crop(bbox)
    return im

image_list = []
for pdf in pdf_files:
    file_path = os.path.join(directory_name, pdf)

    images = convert_from_path(file_path, dpi=150, fmt="png")
    images = [trim_whitespace(img) for img in images]
    final_image = image_concat(images)

    final_image.save(f"{img_directory_name}/{pdf.rsplit('.', 1)[0]}.png", "PNG")
    image_list.append(images)

In [5]:
# Use a raw string with double-braces for the JSON structure
def prompter(ocr_output):
    return f"""
    Extract and structure the information from the following OCR text into a clean JSON format.

    ### OCR TEXT START ###
    {ocr_output}
    ### OCR TEXT END ###

    ### INSTRUCTIONS ###
    1. Clean up OCR errors and normalize text.
    2. For 'Work Experience' and 'Education', concatenate all descriptions/details into a single coherent paragraph for each entry.
    3. If information is missing, use an empty string "" or an empty list [].
    4. Output ONLY valid JSON. Do not include introductory text.

    ### TARGET JSON STRUCTURE ###
    {{
        "personal_info": {{
            "first_name": "",
            "last_name": "",
            "email": "",
            "phone": "",
            "address": "",
            "city": "",
            "country": ""
        }},
        "experiences": [
            {{
                "job_title": "",
                "company": "",
                "dates": "",
                "responsibilities": ""
            }}
        ],
        "education": [
            {{
                "degree": "",
                "institution": "",
                "graduation_date": "",
                "details": ""
            }}
        ],
        "skills": {{
            "technical": "",
            "soft": "",
            "interests": ""
        }}
    }}
    """

In [ ]:

for f in os.listdir(img_directory_name):

    time_start = datetime.datetime.now()
    print(f"Processing {f}")

    # OCR
    tesseract_output = pytesseract.image_to_string(f'{img_directory_name}/{f}')

    time_end = datetime.datetime.now()
    minutes = divmod((time_end - time_start).total_seconds(), 60)
    print(f"OCR Model Execution time is {minutes[0]} minutes and {minutes[1]} seconds.")

    # LLM inference
    output = llm.create_chat_completion(
        messages=[
        ChatCompletionRequestUserMessage(
            role="user",
            content=prompter(tesseract_output) + " /no_think"
        )
        ],
        max_tokens=1536,
        temperature=0.1,   # low temp for structured output
        # enable_thinking=False
        #n_batch=512,
    )

    content = output["choices"][0]["message"]["content"].strip().split("</think>")

    time_end = datetime.datetime.now()
    minutes = divmod((time_end - time_start).total_seconds(), 60)
    print(f"LLM Model Execution time is {minutes[0]} minutes and {minutes[1]} seconds.")

    # Save output
    output_path = f"sample_output_tesseract/{f.rsplit('.', 1)[0]}.json"
    with open(output_path, "w", encoding="utf-8") as file_write:
        file_write.write(content)
    print(f"Saved to {output_path}")

Processing Afundar_AudrieLex_CV.png
OCR Model Execution time is 0.0 minutes and 2.162068 seconds.
